**Ingest race.csv file**
1. Read file using spark dataframe reader API
2.add Metadata Columns 
-   source file
- ingestion Timestamp
3. write to bonze delta table

In [0]:
%run ../00.Common/01.environment-config


In [0]:
%run ../00.Common/02.bronze-helpers


In [0]:
source_file = f"{landing_folder_path}/races.csv"
table_name = f"{catalog_nameone}.{bronze_schema}.races"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StringType, StructField, IntegerType, DateType

race_schema = StructType([
    StructField('season', IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('raceName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType())
  ])

race_df = (
  spark.read.format("csv")
       .option("header", "true")
       .option("mode", "FAILFAST")
       .schema(race_schema)
       .load(source_file)
)

race_final_df = add_ingestion_metadata(race_df)
display(race_final_df)


In [0]:
race_final_df.write.format('delta').mode('overwrite').saveAsTable(table_name)